# SurfaceTarget-CRC — phase 01 — patch 1 16

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.16
#
# FULL TCGA RNA-SEQ DOWNLOAD
# AND DATA INTEGRITY VERIFICATION
#
# Objectives:
# 1. Load selected tumor/normal manifests
# 2. Validate file metadata and access permissions
# 3. Download selected STAR Counts expression files
# 4. Reuse verified existing files
# 5. Verify file size and MD5 checksum
# 6. Maintain resumable download records
# 7. Generate final download summary
#
# IMPORTANT:
# No differential expression analysis is performed.
# No new biological classifications are assigned.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests
import hashlib
import shutil
import json
import time
import os
import uuid

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RAW = ROOT / "data/raw/tcga_star_counts"

RESULTS = ROOT / "results/tables"

TUMOR_DIR = RAW / "tumor"

NORMAL_DIR = RAW / "normal"

for directory in [
    META,
    RAW,
    RESULTS,
    TUMOR_DIR,
    NORMAL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

TUMOR_MANIFEST = (
    META / "tcga_selected_tumor_rnaseq_manifest.csv"
)

NORMAL_MANIFEST = (
    META / "tcga_normal_rnaseq_manifest.csv"
)

DOWNLOAD_LOG = (
    META / "tcga_rnaseq_download_log.csv"
)

GDC_DATA_API = "https://api.gdc.cancer.gov/data"

# Set to None to process the complete dataset.
# Set to an integer, e.g. 20, for a smaller test batch.

MAX_NEW_DOWNLOADS = None

MAX_ATTEMPTS = 4

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.16 — FULL RNA-SEQ DOWNLOAD")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load manifests
# ------------------------------------------------------------

for path in [
    TUMOR_MANIFEST,
    NORMAL_MANIFEST
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required manifest missing: {path}"
        )

tumor = pd.read_csv(
    TUMOR_MANIFEST,
    dtype=str,
    keep_default_na=False
)

normal = pd.read_csv(
    NORMAL_MANIFEST,
    dtype=str,
    keep_default_na=False
)

tumor["dataset_group"] = "Tumor"

normal["dataset_group"] = "Normal"

manifest = pd.concat(
    [tumor, normal],
    ignore_index=True
)

print("\nManifest loaded successfully.")

print("Selected tumor files:", len(tumor))

print("Normal tissue files:", len(normal))

print("Total files:", len(manifest))


# ------------------------------------------------------------
# 3. Validate manifests
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MANIFEST VALIDATION")
print("-" * 65)

required_columns = [
    "file_id",
    "file_size_bytes",
    "md5sum",
    "access",
    "project",
    "patient_id",
    "sample_id"
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest.columns
]

if missing_columns:

    raise RuntimeError(
        f"Missing columns: {missing_columns}"
    )

if len(tumor) != 421:

    raise RuntimeError(
        "Unexpected tumor file count."
    )

if len(normal) != 51:

    raise RuntimeError(
        "Unexpected normal file count."
    )

if manifest["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate file UUIDs detected."
    )

# Validate UUID formatting.

for file_id in manifest["file_id"]:

    try:

        if str(uuid.UUID(file_id)) != file_id.lower():

            raise ValueError("Noncanonical UUID")

    except ValueError:

        raise RuntimeError(
            f"Invalid GDC file UUID: {file_id}"
        )

# Validate access.

access_values = (
    manifest["access"]
    .str.strip()
    .str.lower()
)

if not access_values.eq("open").all():

    restricted = manifest.loc[
        ~access_values.eq("open"),
        ["file_id", "access"]
    ]

    raise RuntimeError(
        "Some files are not marked open-access:\n"
        + restricted.to_string(index=False)
    )

# Validate expected file sizes.

manifest["file_size_bytes"] = pd.to_numeric(
    manifest["file_size_bytes"],
    errors="raise"
).astype("int64")

if (
    manifest["file_size_bytes"] <= 0
).any():

    raise RuntimeError(
        "Invalid expected file sizes."
    )

# Validate MD5 formatting.

manifest["md5sum"] = (
    manifest["md5sum"]
    .str.strip()
    .str.lower()
)

valid_md5 = (
    manifest["md5sum"]
    .str.fullmatch(r"[0-9a-f]{32}")
)

if not valid_md5.all():

    raise RuntimeError(
        "Invalid MD5 checksums in manifest."
    )

print("\nFile count validation: PASSED")
print("Unique UUID validation: PASSED")
print("Open-access validation: PASSED")
print("File size validation: PASSED")
print("MD5 metadata validation: PASSED")


# ------------------------------------------------------------
# 4. Check available Colab storage
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("STORAGE ASSESSMENT")
print("-" * 65)

expected_bytes = int(
    manifest["file_size_bytes"].sum()
)

expected_gib = (
    expected_bytes / (1024 ** 3)
)

disk = shutil.disk_usage("/content")

free_gib = (
    disk.free / (1024 ** 3)
)

print(
    f"\nExpected total dataset: "
    f"{expected_gib:.3f} GiB"
)

print(
    f"Available Colab storage: "
    f"{free_gib:.2f} GiB"
)

# Conservative free-space requirement.

if disk.free < expected_bytes * 2:

    raise RuntimeError(
        "Insufficient free disk space "
        "for safe download."
    )

print("\nStorage assessment: PASSED")


# ------------------------------------------------------------
# 5. MD5 verification functions
# ------------------------------------------------------------

def calculate_md5(file_path):

    md5 = hashlib.md5()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


def verify_file(
    file_path,
    expected_size,
    expected_md5
):

    if not file_path.exists():
        return False

    if (
        file_path.stat().st_size
        != expected_size
    ):
        return False

    actual_md5 = calculate_md5(
        file_path
    )

    return (
        actual_md5.lower()
        == expected_md5.lower()
    )


# ------------------------------------------------------------
# 6. Initialize GDC download session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})


# ------------------------------------------------------------
# 7. Download individual GDC files
# ------------------------------------------------------------

def download_file(
    file_id,
    destination,
    expected_size,
    expected_md5
):

    # Reuse previously verified files.

    if verify_file(
        destination,
        expected_size,
        expected_md5
    ):

        return "Already verified"

    url = f"{GDC_DATA_API}/{file_id}"

    temporary = Path(
        str(destination) + ".part"
    )

    for attempt in range(
        1,
        MAX_ATTEMPTS + 1
    ):

        try:

            if temporary.exists():
                temporary.unlink()

            with session.get(
                url,
                stream=True,
                timeout=(30, 180)
            ) as response:

                response.raise_for_status()

                with open(
                    temporary,
                    "wb"
                ) as output:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if chunk:

                            output.write(chunk)

            # Accept only verified downloads.

            if not verify_file(
                temporary,
                expected_size,
                expected_md5
            ):

                raise RuntimeError(
                    "Size or MD5 verification failed."
                )

            os.replace(
                temporary,
                destination
            )

            return "Downloaded and verified"

        except Exception:

            if temporary.exists():
                temporary.unlink()

            if attempt == MAX_ATTEMPTS:
                raise

            time.sleep(
                min(2 ** attempt, 30)
            )


# ------------------------------------------------------------
# 8. Reuse files downloaded during Patch 1.12
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REUSING PREVIOUSLY VERIFIED FILES")
print("-" * 65)

DUPLICATE_QC_DIR = (
    ROOT / "data/raw/tcga_duplicate_qc"
)

reused_count = 0

for _, row in manifest.iterrows():

    file_id = row["file_id"]

    expected_size = int(
        row["file_size_bytes"]
    )

    expected_md5 = row["md5sum"]

    destination_directory = (
        TUMOR_DIR
        if row["dataset_group"] == "Tumor"
        else NORMAL_DIR
    )

    destination = (
        destination_directory /
        f"{file_id}.tsv"
    )

    previous = (
        DUPLICATE_QC_DIR /
        f"{file_id}.tsv"
    )

    if verify_file(
        destination,
        expected_size,
        expected_md5
    ):
        continue

    if verify_file(
        previous,
        expected_size,
        expected_md5
    ):

        shutil.copy2(
            previous,
            destination
        )

        if not verify_file(
            destination,
            expected_size,
            expected_md5
        ):

            raise RuntimeError(
                "Copied file failed integrity verification."
            )

        reused_count += 1

print(
    "\nPreviously downloaded files reused:",
    reused_count
)


# ------------------------------------------------------------
# 9. Execute full RNA-seq download
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING SELECTED RNA-SEQ DATA")
print("-" * 65)

download_records = []

new_downloads = 0

successful = 0

failed = 0

processed = 0


def save_progress(records):

    progress_df = pd.DataFrame(records)

    temporary_log = Path(
        str(DOWNLOAD_LOG) + ".tmp"
    )

    progress_df.to_csv(
        temporary_log,
        index=False
    )

    os.replace(
        temporary_log,
        DOWNLOAD_LOG
    )


for _, row in manifest.iterrows():

    file_id = row["file_id"]

    dataset_group = row["dataset_group"]

    destination_directory = (
        TUMOR_DIR
        if dataset_group == "Tumor"
        else NORMAL_DIR
    )

    destination = (
        destination_directory /
        f"{file_id}.tsv"
    )

    expected_size = int(
        row["file_size_bytes"]
    )

    expected_md5 = row["md5sum"]

    record = {

        "file_id": file_id,

        "project": row["project"],

        "patient_id": row["patient_id"],

        "sample_id": row["sample_id"],

        "dataset_group": dataset_group,

        "local_path": str(destination),

        "expected_size_bytes": expected_size,

        "expected_md5": expected_md5,

        "status": None,

        "error": None
    }

    try:

        # If a new-download limit was specified,
        # retain already verified files but stop
        # before exceeding that limit.

        already_present = verify_file(
            destination,
            expected_size,
            expected_md5
        )

        if (
            MAX_NEW_DOWNLOADS is not None
            and new_downloads >= MAX_NEW_DOWNLOADS
            and not already_present
        ):

            record["status"] = "Pending"

        else:

            status = download_file(
                file_id,
                destination,
                expected_size,
                expected_md5
            )

            record["status"] = status

            successful += 1

            if status == "Downloaded and verified":
                new_downloads += 1

    except Exception as error:

        failed += 1

        record["status"] = "Failed"

        record["error"] = str(error)

        print(
            f"\nDOWNLOAD FAILED: {file_id}"
        )

        print(error)

    download_records.append(record)

    processed += 1

    # Save progress regularly.

    if processed % 25 == 0:

        save_progress(
            download_records
        )

        print(
            f"\nProgress: {processed}/{len(manifest)} "
            f"| Verified: {successful} "
            f"| Failed: {failed}"
        )

# Save final progress.

save_progress(
    download_records
)

print("\nDownload processing finished.")


# ------------------------------------------------------------
# 10. Verify downloaded dataset
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL DATA INTEGRITY AUDIT")
print("-" * 65)

verification_records = []

for _, row in manifest.iterrows():

    file_id = row["file_id"]

    directory = (
        TUMOR_DIR
        if row["dataset_group"] == "Tumor"
        else NORMAL_DIR
    )

    file_path = (
        directory /
        f"{file_id}.tsv"
    )

    verified = verify_file(
        file_path,
        int(row["file_size_bytes"]),
        row["md5sum"]
    )

    verification_records.append({

        "file_id": file_id,

        "project": row["project"],

        "dataset_group": row["dataset_group"],

        "verified": verified
    })

verification = pd.DataFrame(
    verification_records
)

print("\nOverall verification:")

print(
    verification["verified"]
    .value_counts()
    .to_string()
)

print("\nVerification by dataset:")

verification_summary = (
    verification.groupby(
        ["project", "dataset_group"]
    )
    .agg(
        files=("file_id", "count"),
        verified_files=("verified", "sum")
    )
    .reset_index()
)

print(
    verification_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 11. Save final verification records
# ------------------------------------------------------------

verification_path = (
    META /
    "tcga_rnaseq_download_verification.csv"
)

verification.to_csv(
    verification_path,
    index=False
)

summary_path = (
    RESULTS /
    "tcga_rnaseq_download_summary.csv"
)

verification_summary.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 12. Save research provenance
# ------------------------------------------------------------

all_verified = bool(
    verification["verified"].all()
)

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.16",

    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "endpoint": GDC_DATA_API,

    "workflow": "STAR - Counts",

    "manifest_files": len(manifest),

    "tumor_files": len(tumor),

    "normal_files": len(normal),

    "verified_files": int(
        verification["verified"].sum()
    ),

    "failed_download_attempts": failed,

    "all_files_verified": all_verified,

    "integrity_checks": [
        "File size",
        "MD5 checksum"
    ],

    "count_column_for_downstream_analysis":
        "unstranded",

    "expression_matrix_constructed": False,

    "differential_expression_performed": False,

    "limitations": [
        "File integrity does not establish biological quality.",
        "Full-cohort expression QC remains pending.",
        "Workflow version and evidence tier are strongly associated.",
        "Normal samples are not all patient-matched.",
        "Tier B is computationally low-MSI, not categorically verified MSS."
    ]
}

provenance_path = (
    META /
    "tcga_rnaseq_download_provenance.json"
)

with open(provenance_path, "w") as output:

    json.dump(
        provenance,
        output,
        indent=4
    )


# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.16 SUMMARY")
print("=" * 65)

print(
    "\nTotal manifest files:",
    len(manifest)
)

print(
    "Successfully verified:",
    int(verification["verified"].sum())
)

print(
    "Files not verified:",
    int((~verification["verified"]).sum())
)

print(
    "Download attempts that failed:",
    failed
)

print(
    "\nAll files verified:",
    all_verified
)

print("\nOutput locations:")

print("Tumor:", TUMOR_DIR)

print("Normal:", NORMAL_DIR)

print("\nSaved metadata:")

print("1. tcga_rnaseq_download_log.csv")

print("2. tcga_rnaseq_download_verification.csv")

print("3. tcga_rnaseq_download_summary.csv")

print("4. tcga_rnaseq_download_provenance.json")

if all_verified:

    print("\nALL RNA-SEQ FILES VERIFIED SUCCESSFULLY")

    print("\nPATCH 1.16 COMPLETE")

else:

    print(
        "\nDownload incomplete. "
        "Rerun this patch to retry missing files."
    )

print("=" * 65)